In [1]:
pip install scikit-learn pandas numpy joblib


  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
     ---------------------------------------- 9.9/9.9 MB 9.0 MB/s eta 0:00:00
     --------------------------------------- 12.6/12.6 MB 13.4 MB/s eta 0:00:00
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
     ---------------------------------------- 36.6/36.6 MB 9.9 MB/s eta 0:00:00
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
     ------------------------------------- 349.0/349.0 kB 10.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
"""
SAMVED – Smart Safety System for Sewer Workers
================================================
STEP 1: Synthetic Dataset Generator

Generates a realistic training dataset for the ML-based risk classifier.
Based on OSHA gas exposure limits and physiological response benchmarks.

Classes:
  0 = SAFE     – all parameters normal, trends stable
  1 = WARNING  – rising trends or mild compound risk
  2 = DANGER   – critical levels or fast-deteriorating multi-signal trend

Run:
  python samved_generate_dataset.py
"""

import numpy as np
import pandas as pd

np.random.seed(42)
SAMPLES_PER_CLASS = 1000  # 3000 total rows, balanced

# ──────────────────────────────────────────────
# Reference limits (used to frame value ranges)
# ──────────────────────────────────────────────
# CH4   – LEL (lower explosive limit) = 5% = 50,000 ppm
#         Concern zone: >1,000 ppm (2% LEL)
# H2S   – IDLH = 100 ppm | OSHA PEL = 20 ppm ceiling
# CO    – IDLH = 1200 ppm | OSHA PEL = 50 ppm (TWA)
# O2    – Normal = 20.9% | Deficiency < 19.5% | IDLH < 16%
# HR    – Normal rest = 60–100 bpm | Stress = 100–140 bpm
# SpO2  – Normal = 96–100% | Low = 90–95% | Critical < 90%


def _clamp(val, lo, hi):
    return float(np.clip(val, lo, hi))


def generate_safe():
    """All parameters in normal operating range. Trends flat."""
    ch4       = _clamp(np.random.normal(200,  100),    0,    900)
    h2s       = _clamp(np.random.normal(2,    1.5),    0,    8)
    co        = _clamp(np.random.normal(10,   6),      0,    35)
    o2        = _clamp(np.random.normal(20.5, 0.3),   19.6,  21.0)
    hr        = _clamp(np.random.normal(78,   8),     60,    95)
    spo2      = _clamp(np.random.normal(98,   0.8),   96,    100)
    # Deltas: nearly flat
    ch4_d     = np.random.uniform(-15,   15)
    h2s_d     = np.random.uniform(-0.3,  0.3)
    o2_d      = np.random.uniform(-0.05, 0.05)
    hr_d      = np.random.uniform(-2,    2)
    spo2_d    = np.random.uniform(-0.2,  0.2)
    motion    = np.random.choice([0, 1], p=[0.3, 0.7])  # mostly moving
    return [ch4, h2s, co, o2, hr, spo2, ch4_d, h2s_d, o2_d, hr_d, spo2_d, motion, 0]


def generate_warning():
    """
    Two sub-scenarios mixed:
      A) Elevated gas levels but vitals still OK (single-signal warning)
      B) Mild vitals stress with gentle rising gas trend (compound early warning)
    """
    scenario = np.random.choice(['A', 'B'])

    if scenario == 'A':
        ch4   = _clamp(np.random.normal(1800, 400),  900,  4000)
        h2s   = _clamp(np.random.normal(12,   5),    6,    25)
        co    = _clamp(np.random.normal(45,   15),   20,   90)
        o2    = _clamp(np.random.normal(19.0, 0.4), 18.0,  19.5)
        hr    = _clamp(np.random.normal(88,   8),   78,    105)
        spo2  = _clamp(np.random.normal(95,   1),   92,    97)
        ch4_d = np.random.uniform(10,   80)   # rising, not alarming
        h2s_d = np.random.uniform(0.5,  3)
        o2_d  = np.random.uniform(-0.2, -0.02)
        hr_d  = np.random.uniform(1,    6)
        spo2_d= np.random.uniform(-0.5, 0)
        motion= np.random.choice([0, 1], p=[0.5, 0.5])

    else:  # B – compound early warning
        ch4   = _clamp(np.random.normal(600,  200),  200,  1500)
        h2s   = _clamp(np.random.normal(8,    3),    3,    18)
        co    = _clamp(np.random.normal(30,   10),   10,   65)
        o2    = _clamp(np.random.normal(19.3, 0.5), 18.2,  19.8)
        hr    = _clamp(np.random.normal(100,  8),   88,    118)
        spo2  = _clamp(np.random.normal(94,   1.5), 91,    97)
        ch4_d = np.random.uniform(20,   100)
        h2s_d = np.random.uniform(1,    5)
        o2_d  = np.random.uniform(-0.4, -0.1)   # notable drop
        hr_d  = np.random.uniform(4,    12)      # stress rising
        spo2_d= np.random.uniform(-1,   -0.2)
        motion= np.random.choice([0, 1], p=[0.6, 0.4])

    return [ch4, h2s, co, o2, hr, spo2, ch4_d, h2s_d, o2_d, hr_d, spo2_d, motion, 1]


def generate_danger():
    """
    Three sub-scenarios:
      A) Critical gas levels regardless of vitals
      B) Multi-signal deterioration (gas + vitals collapsing together)
      C) Fall detected with deteriorating environment
    """
    scenario = np.random.choice(['A', 'B', 'C'], p=[0.35, 0.45, 0.20])

    if scenario == 'A':  # gas critical
        ch4   = _clamp(np.random.normal(4000, 600),  2500,  6000)
        h2s   = _clamp(np.random.normal(55,   20),   25,    100)
        co    = _clamp(np.random.normal(180,  50),   100,   400)
        o2    = _clamp(np.random.normal(16.5, 1.0),  13.0,  18.5)
        hr    = _clamp(np.random.normal(115,  12),   100,   145)
        spo2  = _clamp(np.random.normal(91,   2),    85,    94)
        ch4_d = np.random.uniform(80,   300)
        h2s_d = np.random.uniform(5,    20)
        o2_d  = np.random.uniform(-1.2, -0.4)
        hr_d  = np.random.uniform(10,   25)
        spo2_d= np.random.uniform(-2,   -0.5)
        motion= np.random.choice([0, 1], p=[0.6, 0.4])

    elif scenario == 'B':  # multi-signal collapse
        ch4   = _clamp(np.random.normal(2500, 500),  1200,  5000)
        h2s   = _clamp(np.random.normal(35,   15),   15,    80)
        co    = _clamp(np.random.normal(120,  40),   60,    300)
        o2    = _clamp(np.random.normal(15.5, 1.2),  13.0,  18.0)
        hr    = _clamp(np.random.normal(130,  10),   110,   155)
        spo2  = _clamp(np.random.normal(88,   2),    82,    92)
        ch4_d = np.random.uniform(50,   200)
        h2s_d = np.random.uniform(3,    15)
        o2_d  = np.random.uniform(-1.5, -0.5)
        hr_d  = np.random.uniform(15,   35)
        spo2_d= np.random.uniform(-3,   -1)
        motion= np.random.choice([0, 1], p=[0.7, 0.3])

    else:  # C – fall + bad environment
        ch4   = _clamp(np.random.normal(1800, 600),  600,   4000)
        h2s   = _clamp(np.random.normal(25,   10),   8,     60)
        co    = _clamp(np.random.normal(90,   30),   30,    200)
        o2    = _clamp(np.random.normal(17.5, 1.0),  14.0,  19.0)
        hr    = _clamp(np.random.normal(120,  15),   95,    150)
        spo2  = _clamp(np.random.normal(90,   2),    84,    95)
        ch4_d = np.random.uniform(20,   120)
        h2s_d = np.random.uniform(2,    10)
        o2_d  = np.random.uniform(-1.0, -0.3)
        hr_d  = np.random.uniform(8,    30)
        spo2_d= np.random.uniform(-2,   -0.5)
        motion= 2  # fall detected — always 2 in this scenario

    return [ch4, h2s, co, o2, hr, spo2, ch4_d, h2s_d, o2_d, hr_d, spo2_d, motion, 2]


# ──────────────────────────────────────────────
# Build dataset
# ──────────────────────────────────────────────
generators = {0: generate_safe, 1: generate_warning, 2: generate_danger}
rows = []

for label, fn in generators.items():
    for _ in range(SAMPLES_PER_CLASS):
        rows.append(fn())

columns = [
    'ch4_ppm', 'h2s_ppm', 'co_ppm', 'o2_percent',
    'heart_rate_bpm', 'spo2_percent',
    'ch4_delta', 'h2s_delta', 'o2_delta', 'hr_delta', 'spo2_delta',
    'motion_state',   # 0=stationary, 1=moving, 2=fall
    'label'           # 0=SAFE, 1=WARNING, 2=DANGER
]

df = pd.DataFrame(rows, columns=columns)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.to_csv('samved_dataset.csv', index=False)

print("=" * 55)
print("  SAMVED – Dataset Generation Complete")
print("=" * 55)
print(f"  Total samples : {len(df)}")
print(f"  Features      : {len(columns) - 1}")
print()
print("  Class distribution:")
label_map = {0: 'SAFE', 1: 'WARNING', 2: 'DANGER'}
for lbl, name in label_map.items():
    count = (df['label'] == lbl).sum()
    print(f"    {name:8s} ({lbl}) : {count}")
print()
print("  Feature statistics:")
print(df.drop('label', axis=1).describe().round(2).to_string())
print()
print("  Saved → samved_dataset.csv")


  SAMVED – Dataset Generation Complete
  Total samples : 3000
  Features      : 12

  Class distribution:
    SAFE     (0) : 1000
    WARNING  (1) : 1000
    DANGER   (2) : 1000

  Feature statistics:
       ch4_ppm  h2s_ppm   co_ppm  o2_percent  heart_rate_bpm  spo2_percent  ch4_delta  h2s_delta  o2_delta  hr_delta  spo2_delta  motion_state
count  3000.00  3000.00  3000.00     3000.00         3000.00       3000.00    3000.00    3000.00   3000.00   3000.00     3000.00       3000.00
mean   1449.32    17.74    63.20       18.64           98.17         93.99      63.20       3.99     -0.34      8.87       -0.65          0.61
std    1324.59    20.27    64.89        1.93           20.97          3.88      69.67       4.83      0.40      9.76        0.77          0.61
min       0.00     0.00     0.00       13.00           60.00         83.01     -14.93      -0.30     -1.50     -2.00       -2.99          0.00
25%     254.08     2.97    14.09       17.17           81.20         91.18       7.4

In [2]:
"""
SAMVED – Smart Safety System for Sewer Workers
================================================
STEP 2: Model Training, Evaluation & Export

Trains a Random Forest classifier on the synthetic SAMVED dataset.
Outputs:
  - samved_model.pkl        → trained model (used by inference script)
  - samved_scaler.pkl       → feature scaler (not strictly needed for RF,
                              included for easy swap to other models)
  - samved_feature_names.pkl → column order (required for inference)
  - samved_training_report.txt → full evaluation report

Run AFTER samved_generate_dataset.py:
  python samved_train_model.py
"""

import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.preprocessing import StandardScaler

# ──────────────────────────────────────────────
# 1. Load data
# ──────────────────────────────────────────────
print("\nSAMVED – Model Training")
print("=" * 55)

df = pd.read_csv('samved_dataset.csv')
label_map = {0: 'SAFE', 1: 'WARNING', 2: 'DANGER'}
print(f"Loaded {len(df)} samples")

FEATURES = [
    'ch4_ppm', 'h2s_ppm', 'co_ppm', 'o2_percent',
    'heart_rate_bpm', 'spo2_percent',
    'ch4_delta', 'h2s_delta', 'o2_delta', 'hr_delta', 'spo2_delta',
    'motion_state'
]

X = df[FEATURES]
y = df['label']

# ──────────────────────────────────────────────
# 2. Train / test split
# ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# Scaler (stored for pipeline completeness; RF doesn't need it)
scaler = StandardScaler()
scaler.fit(X_train)
joblib.dump(scaler, 'samved_scaler.pkl')

# ──────────────────────────────────────────────
# 3. Train Random Forest
# ──────────────────────────────────────────────
# Why Random Forest for SAMVED?
#   ✓ Handles mixed numeric + categorical (motion_state) naturally
#   ✓ Feature importance – shows judges WHICH signals drive risk
#   ✓ No normalization required
#   ✓ Robust to small dataset / synthetic data
#   ✓ Fast inference – can run on Raspberry Pi or companion app easily
#   ✓ predict_proba() gives risk score % without extra work

model = RandomForestClassifier(
    n_estimators=100,   # 100 trees – good balance for a prototype
    max_depth=10,       # prevent overfitting on synthetic data
    min_samples_leaf=4,
    random_state=42,
    class_weight='balanced'  # guards against edge imbalance
)
model.fit(X_train, y_train)
print("Model trained ✓")

# ──────────────────────────────────────────────
# 4. Evaluation
# ──────────────────────────────────────────────
y_pred = model.predict(X_test)
y_pred_names = [label_map[p] for p in y_pred]
y_test_names = [label_map[t] for t in y_test]

accuracy = accuracy_score(y_test, y_pred)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print(f"\nTest Accuracy   : {accuracy * 100:.2f}%")
print(f"5-Fold CV Mean  : {cv_scores.mean() * 100:.2f}% ± {cv_scores.std() * 100:.2f}%")

# ──────────────────────────────────────────────
# 5. Feature Importance (KEY for judge demo)
# ──────────────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=FEATURES)
importances_sorted = importances.sort_values(ascending=False)

print("\nFeature Importance (most → least influential):")
for feat, imp in importances_sorted.items():
    bar = '█' * int(imp * 60)
    print(f"  {feat:20s}  {imp:.4f}  {bar}")

# ──────────────────────────────────────────────
# 6. Confusion Matrix
# ──────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(f"{'':10s}  {'SAFE':>8s}  {'WARNING':>8s}  {'DANGER':>8s}")
for i, row_label in enumerate(['SAFE', 'WARNING', 'DANGER']):
    print(f"  {row_label:8s}  {cm[i][0]:8d}  {cm[i][1]:8d}  {cm[i][2]:8d}")

# ──────────────────────────────────────────────
# 7. Full classification report
# ──────────────────────────────────────────────
report = classification_report(y_test_names, y_pred_names,
                                target_names=['SAFE', 'WARNING', 'DANGER'])
print("\nClassification Report:")
print(report)

# ──────────────────────────────────────────────
# 8. Demo – show what predict_proba looks like
# ──────────────────────────────────────────────
print("Risk Score Demo (predict_proba on 3 example inputs):")

examples = {
    'Normal shift start (SAFE)': {
        'ch4_ppm': 150, 'h2s_ppm': 1.5, 'co_ppm': 8, 'o2_percent': 20.6,
        'heart_rate_bpm': 75, 'spo2_percent': 98,
        'ch4_delta': 5, 'h2s_delta': 0.1, 'o2_delta': 0.01,
        'hr_delta': 1, 'spo2_delta': 0.0, 'motion_state': 1
    },
    'Gas rising, mild stress (WARNING)': {
        'ch4_ppm': 1500, 'h2s_ppm': 10, 'co_ppm': 40, 'o2_percent': 19.1,
        'heart_rate_bpm': 97, 'spo2_percent': 94,
        'ch4_delta': 60, 'h2s_delta': 2.0, 'o2_delta': -0.2,
        'hr_delta': 6, 'spo2_delta': -0.5, 'motion_state': 1
    },
    'Critical multi-signal collapse (DANGER)': {
        'ch4_ppm': 4200, 'h2s_ppm': 60, 'co_ppm': 200, 'o2_percent': 15.5,
        'heart_rate_bpm': 135, 'spo2_percent': 87,
        'ch4_delta': 180, 'h2s_delta': 12, 'o2_delta': -1.1,
        'hr_delta': 22, 'spo2_delta': -2.5, 'motion_state': 2
    }
}

for scenario_name, vals in examples.items():
    sample = pd.DataFrame([vals])[FEATURES]
    proba = model.predict_proba(sample)[0]
    predicted_label = label_map[model.predict(sample)[0]]
    risk_score = int(proba[2] * 100)  # P(DANGER) as risk %
    print(f"\n  Scenario: {scenario_name}")
    print(f"    P(SAFE)={proba[0]:.3f}  P(WARNING)={proba[1]:.3f}  P(DANGER)={proba[2]:.3f}")
    print(f"    → Predicted: {predicted_label}  |  Risk Score: {risk_score}%")

# ──────────────────────────────────────────────
# 9. Save model + metadata
# ──────────────────────────────────────────────
joblib.dump(model, 'samved_model.pkl')
joblib.dump(FEATURES, 'samved_feature_names.pkl')

# Write a human-readable report
report_lines = [
    "SAMVED – ML Training Report",
    "=" * 55,
    f"Test Accuracy  : {accuracy * 100:.2f}%",
    f"CV Mean        : {cv_scores.mean() * 100:.2f}% ± {cv_scores.std() * 100:.2f}%",
    "",
    "Feature Importance:",
    *[f"  {f:20s}  {v:.4f}" for f, v in importances_sorted.items()],
    "",
    "Classification Report:",
    report
]
with open('samved_training_report.txt', 'w') as f:
    f.write('\n'.join(report_lines))

print("\n" + "=" * 55)
print("  Saved:")
print("    samved_model.pkl")
print("    samved_scaler.pkl")
print("    samved_feature_names.pkl")
print("    samved_training_report.txt")
print("=" * 55)



SAMVED – Model Training
Loaded 3000 samples
Train: 2400 | Test: 600
Model trained ✓

Test Accuracy   : 100.00%
5-Fold CV Mean  : 100.00% ± 0.00%

Feature Importance (most → least influential):
  h2s_delta             0.1973  ███████████
  o2_delta              0.1882  ███████████
  o2_percent            0.1510  █████████
  hr_delta              0.1068  ██████
  spo2_percent          0.0868  █████
  h2s_ppm               0.0863  █████
  co_ppm                0.0696  ████
  ch4_delta             0.0641  ███
  ch4_ppm               0.0233  █
  heart_rate_bpm        0.0137  
  spo2_delta            0.0069  
  motion_state          0.0061  

Confusion Matrix (rows=actual, cols=predicted):
                SAFE   WARNING    DANGER
  SAFE           200         0         0
  WARNING          0       200         0
  DANGER           0         0       200

Classification Report:
              precision    recall  f1-score   support

        SAFE       1.00      1.00      1.00       200
     WARN

In [3]:
"""
SAMVED – Smart Safety System for Sewer Workers
================================================
STEP 3: Real-Time Inference Engine

This module is what you ACTUALLY CALL from your dashboard backend,
Firebase cloud function, or local Python server that receives
ESP32 sensor data and needs to produce a risk decision.

Usage:
  from samved_inference import SAMVEDInference
  engine = SAMVEDInference()
  result = engine.predict(sensor_reading)
  print(result)

Also runnable standalone for testing:
  python samved_inference.py
"""

import joblib
import pandas as pd
import time
from collections import deque


# ──────────────────────────────────────────────
# Constants
# ──────────────────────────────────────────────
LABEL_MAP = {0: 'SAFE', 1: 'WARNING', 2: 'DANGER'}

RISK_THRESHOLDS = {
    'SAFE':    (0,  35),
    'WARNING': (35, 65),
    'DANGER':  (65, 100)
}

ACTIONS = {
    'SAFE':    'Continue work. Monitor regularly.',
    'WARNING': 'Caution. Supervisor alerted. Prepare to exit.',
    'DANGER':  'EXIT IMMEDIATELY. Emergency protocol activated.'
}

TREND_LABELS = {
    'stable':   'Stable',
    'rising':   'Rising – monitor closely',
    'critical': 'Critical – rapid deterioration'
}

# Hard override thresholds (bypass ML – instant DANGER)
# These are non-negotiable physical safety limits.
HARD_LIMITS = {
    'o2_percent':      16.0,   # below this = DANGER regardless of ML
    'h2s_ppm':         50.0,   # IDLH threshold
    'co_ppm':          200.0,  # approaching IDLH
    'ch4_ppm':         5000.0, # 10% LEL – explosive risk zone
    'spo2_percent':    90.0,   # critical hypoxia
    'motion_state':    2       # fall detected = always DANGER
}


class SAMVEDInference:
    """
    SAMVED real-time risk inference engine.

    Wraps the trained Random Forest model with:
      - hard-limit override logic (fail-safe)
      - multi-step trend tracking (sliding window)
      - risk score as P(DANGER) probability
      - structured JSON-ready output for dashboard/Firebase
    """

    def __init__(self, model_path='samved_model.pkl',
                 features_path='samved_feature_names.pkl',
                 window_size=5):
        """
        Args:
            model_path    : path to trained model pkl
            features_path : path to feature names pkl
            window_size   : N readings used for trend analysis
        """
        self.model = joblib.load(model_path)
        self.features = joblib.load(features_path)
        self.window = deque(maxlen=window_size)  # last N risk scores
        self._override_reason = None

    # ──────────────────────────────────────────────
    # Public API
    # ──────────────────────────────────────────────
    def predict(self, reading: dict) -> dict:
        """
        Main inference function.

        Args:
            reading: dict with keys matching FEATURES list.
                     Must include the 12 features.
                     ch4_delta, h2s_delta etc. should be computed
                     by the caller as (current_val - previous_val).

        Returns:
            dict with all dashboard-ready fields.
        """
        self._override_reason = None

        # 1. Check hard-limit overrides first
        override = self._check_hard_limits(reading)

        if override:
            label = 'DANGER'
            risk_score = 100
            confidence = {'SAFE': 0.0, 'WARNING': 0.0, 'DANGER': 1.0}
        else:
            # 2. ML inference
            sample = pd.DataFrame([reading])[self.features]
            proba = self.model.predict_proba(sample)[0]
            predicted_class = self.model.predict(sample)[0]
            label = LABEL_MAP[predicted_class]

            confidence = {
                'SAFE':    round(float(proba[0]), 3),
                'WARNING': round(float(proba[1]), 3),
                'DANGER':  round(float(proba[2]), 3)
            }
            # Risk score = P(DANGER) scaled to 0–100
            risk_score = int(proba[2] * 100)

        # 3. Trend analysis from sliding window
        self.window.append(risk_score)
        trend = self._compute_trend()

        # 4. Build full output
        result = {
            # ── Core decision ──
            'status':           label,
            'risk_score':       risk_score,
            'trend':            trend,
            'action':           ACTIONS[label],

            # ── Confidence breakdown ──
            'confidence':       confidence,

            # ── Override info ──
            'hard_limit_triggered': override,
            'override_reason':  self._override_reason,

            # ── Alert flags (used to trigger buzzer / SOS) ──
            'alert_local_buzzer':   label == 'DANGER',
            'alert_supervisor':     label in ('WARNING', 'DANGER'),
            'alert_sos':            label == 'DANGER' and override,
            'alert_fall':           reading.get('motion_state') == 2,

            # ── Dashboard display fields ──
            'display': {
                'status_color':     self._status_color(label),
                'trend_label':      TREND_LABELS.get(trend, trend),
                'risk_bar':         risk_score,      # 0–100 for progress bar
                'recommended_action': ACTIONS[label]
            },

            # ── Timestamp ──
            'timestamp_ms': int(time.time() * 1000)
        }

        return result

    # ──────────────────────────────────────────────
    # Internal helpers
    # ──────────────────────────────────────────────
    def _check_hard_limits(self, reading: dict) -> bool:
        """
        Returns True if any hard limit is breached.
        Sets self._override_reason to human-readable string.
        """
        checks = [
            (reading.get('o2_percent', 21),       '<=', HARD_LIMITS['o2_percent'],   'O2 critically low'),
            (reading.get('h2s_ppm', 0),            '>=', HARD_LIMITS['h2s_ppm'],     'H2S at IDLH level'),
            (reading.get('co_ppm', 0),             '>=', HARD_LIMITS['co_ppm'],      'CO approaching IDLH'),
            (reading.get('ch4_ppm', 0),            '>=', HARD_LIMITS['ch4_ppm'],     'CH4 explosive risk zone'),
            (reading.get('spo2_percent', 100),     '<=', HARD_LIMITS['spo2_percent'],'SpO2 critically low – hypoxia'),
            (reading.get('motion_state', 0),       '==', HARD_LIMITS['motion_state'],'Fall detected'),
        ]
        for val, op, limit, reason in checks:
            triggered = (
                (op == '<=' and val <= limit) or
                (op == '>=' and val >= limit) or
                (op == '==' and val == limit)
            )
            if triggered:
                self._override_reason = reason
                return True
        return False

    def _compute_trend(self) -> str:
        """
        Analyses sliding window of risk scores to determine trend.
        Returns: 'stable', 'rising', or 'critical'
        """
        if len(self.window) < 3:
            return 'stable'

        scores = list(self.window)
        recent = scores[-3:]
        delta = recent[-1] - recent[0]

        if delta >= 20:
            return 'critical'
        elif delta >= 8:
            return 'rising'
        else:
            return 'stable'

    @staticmethod
    def _status_color(label: str) -> str:
        return {'SAFE': '#22c55e', 'WARNING': '#f59e0b', 'DANGER': '#ef4444'}[label]


# ──────────────────────────────────────────────
# Standalone test
# ──────────────────────────────────────────────
if __name__ == '__main__':
    engine = SAMVEDInference()

    test_cases = [
        {
            'name': 'Normal start-of-shift reading',
            'data': {
                'ch4_ppm': 120, 'h2s_ppm': 1.2, 'co_ppm': 7, 'o2_percent': 20.7,
                'heart_rate_bpm': 74, 'spo2_percent': 98,
                'ch4_delta': 3, 'h2s_delta': 0.05, 'o2_delta': 0.02,
                'hr_delta': 1, 'spo2_delta': 0.0, 'motion_state': 1
            }
        },
        {
            'name': 'Gas rising steadily – early warning zone',
            'data': {
                'ch4_ppm': 1400, 'h2s_ppm': 9, 'co_ppm': 38, 'o2_percent': 19.2,
                'heart_rate_bpm': 95, 'spo2_percent': 94.5,
                'ch4_delta': 55, 'h2s_delta': 1.8, 'o2_delta': -0.18,
                'hr_delta': 5, 'spo2_delta': -0.4, 'motion_state': 1
            }
        },
        {
            'name': 'Multi-signal critical – ML DANGER',
            'data': {
                'ch4_ppm': 4000, 'h2s_ppm': 45, 'co_ppm': 190, 'o2_percent': 16.2,
                'heart_rate_bpm': 130, 'spo2_percent': 88,
                'ch4_delta': 160, 'h2s_delta': 10, 'o2_delta': -1.0,
                'hr_delta': 20, 'spo2_delta': -2.0, 'motion_state': 1
            }
        },
        {
            'name': 'Fall detected – hard override',
            'data': {
                'ch4_ppm': 800, 'h2s_ppm': 5, 'co_ppm': 20, 'o2_percent': 20.1,
                'heart_rate_bpm': 88, 'spo2_percent': 96,
                'ch4_delta': 10, 'h2s_delta': 0.3, 'o2_delta': -0.05,
                'hr_delta': 3, 'spo2_delta': -0.1, 'motion_state': 2  # FALL
            }
        }
    ]

    print("\nSAMVED – Inference Engine Test")
    print("=" * 60)

    for case in test_cases:
        result = engine.predict(case['data'])
        print(f"\n  Test: {case['name']}")
        print(f"  ├─ Status     : {result['status']}")
        print(f"  ├─ Risk Score : {result['risk_score']}%")
        print(f"  ├─ Trend      : {result['trend']}")
        print(f"  ├─ Confidence : {result['confidence']}")
        print(f"  ├─ Override   : {result['hard_limit_triggered']} "
              f"({result['override_reason'] or 'N/A'})")
        print(f"  ├─ Buzzer     : {'YES' if result['alert_local_buzzer'] else 'no'}")
        print(f"  ├─ SOS        : {'YES' if result['alert_sos'] else 'no'}")
        print(f"  └─ Action     : {result['action']}")

    print("\n" + "=" * 60)
    print("  Inference engine working correctly.")



SAMVED – Inference Engine Test

  Test: Normal start-of-shift reading
  ├─ Status     : SAFE
  ├─ Risk Score : 0%
  ├─ Trend      : stable
  ├─ Confidence : {'SAFE': 1.0, 'WARNING': 0.0, 'DANGER': 0.0}
  ├─ Override   : False (N/A)
  ├─ Buzzer     : no
  ├─ SOS        : no
  └─ Action     : Continue work. Monitor regularly.

  Test: Gas rising steadily – early warning zone
  ├─ Status     : WARNING
  ├─ Risk Score : 0%
  ├─ Trend      : stable
  ├─ Confidence : {'SAFE': 0.0, 'WARNING': 1.0, 'DANGER': 0.0}
  ├─ Override   : False (N/A)
  ├─ Buzzer     : no
  ├─ SOS        : no
  └─ Action     : Caution. Supervisor alerted. Prepare to exit.

  Test: Multi-signal critical – ML DANGER
  ├─ Status     : DANGER
  ├─ Risk Score : 100%
  ├─ Trend      : critical
  ├─ Confidence : {'SAFE': 0.0, 'WARNING': 0.0, 'DANGER': 1.0}
  ├─ Override   : True (SpO2 critically low – hypoxia)
  ├─ Buzzer     : YES
  ├─ SOS        : YES
  └─ Action     : EXIT IMMEDIATELY. Emergency protocol activated.

  Tes

In [4]:
import joblib
import pandas as pd
import time
from collections import deque

# ── Paste SAMVEDInference class here (from samved_inference.py) ──

LABEL_MAP = {0: 'SAFE', 1: 'WARNING', 2: 'DANGER'}
ACTIONS = {
    'SAFE':    'Continue work. Monitor regularly.',
    'WARNING': 'Caution. Supervisor alerted. Prepare to exit.',
    'DANGER':  'EXIT IMMEDIATELY. Emergency protocol activated.'
}
TREND_LABELS = {
    'stable': 'Stable',
    'rising': 'Rising – monitor closely',
    'critical': 'Critical – rapid deterioration'
}
HARD_LIMITS = {
    'o2_percent': 16.0, 'h2s_ppm': 50.0, 'co_ppm': 200.0,
    'ch4_ppm': 5000.0, 'spo2_percent': 90.0, 'motion_state': 2
}

class SAMVEDInference:
    def __init__(self, model_path='samved_model.pkl', features_path='samved_feature_names.pkl', window_size=5):
        self.model = joblib.load(model_path)
        self.features = joblib.load(features_path)
        self.window = deque(maxlen=window_size)
        self._override_reason = None

    def predict(self, reading):
        self._override_reason = None
        override = self._check_hard_limits(reading)
        if override:
            label, risk_score = 'DANGER', 100
            confidence = {'SAFE': 0.0, 'WARNING': 0.0, 'DANGER': 1.0}
        else:
            sample = pd.DataFrame([reading])[self.features]
            proba = self.model.predict_proba(sample)[0]
            label = LABEL_MAP[self.model.predict(sample)[0]]
            confidence = {'SAFE': round(float(proba[0]),3), 'WARNING': round(float(proba[1]),3), 'DANGER': round(float(proba[2]),3)}
            risk_score = int(proba[2] * 100)
        self.window.append(risk_score)
        trend = self._compute_trend()
        return {
            'status': label, 'risk_score': risk_score, 'trend': trend,
            'action': ACTIONS[label], 'confidence': confidence,
            'hard_limit_triggered': override, 'override_reason': self._override_reason,
            'alert_local_buzzer': label == 'DANGER',
            'alert_sos': label == 'DANGER' and override,
            'alert_fall': reading.get('motion_state') == 2,
        }

    def _check_hard_limits(self, reading):
        checks = [
            (reading.get('o2_percent', 21),    '<=', HARD_LIMITS['o2_percent'],    'O2 critically low'),
            (reading.get('h2s_ppm', 0),        '>=', HARD_LIMITS['h2s_ppm'],       'H2S at IDLH level'),
            (reading.get('co_ppm', 0),         '>=', HARD_LIMITS['co_ppm'],        'CO approaching IDLH'),
            (reading.get('ch4_ppm', 0),        '>=', HARD_LIMITS['ch4_ppm'],       'CH4 explosive risk zone'),
            (reading.get('spo2_percent', 100), '<=', HARD_LIMITS['spo2_percent'],  'SpO2 critically low'),
            (reading.get('motion_state', 0),   '==', HARD_LIMITS['motion_state'],  'Fall detected'),
        ]
        for val, op, limit, reason in checks:
            if (op=='<=' and val<=limit) or (op=='>=' and val>=limit) or (op=='==' and val==limit):
                self._override_reason = reason
                return True
        return False

    def _compute_trend(self):
        if len(self.window) < 3: return 'stable'
        scores = list(self.window)
        delta = scores[-1] - scores[-3]
        return 'critical' if delta >= 20 else 'rising' if delta >= 8 else 'stable'


# ── DeltaCalculator ──

class DeltaCalculator:
    DELTA_FIELDS = ['ch4_ppm', 'h2s_ppm', 'o2_percent', 'heart_rate_bpm', 'spo2_percent']
    DELTA_OUTPUT_MAP = {
        'ch4_ppm': 'ch4_delta', 'h2s_ppm': 'h2s_delta',
        'o2_percent': 'o2_delta', 'heart_rate_bpm': 'hr_delta', 'spo2_percent': 'spo2_delta'
    }

    def __init__(self, window=3):
        self._history = {f: deque(maxlen=window) for f in self.DELTA_FIELDS}

    def enrich(self, raw):
        for field in self.DELTA_FIELDS:
            if raw.get(field) is not None:
                self._history[field].append(float(raw[field]))
        enriched = dict(raw)
        for field, delta_key in self.DELTA_OUTPUT_MAP.items():
            hist = self._history[field]
            enriched[delta_key] = hist[-1] - hist[0] if len(hist) >= 2 else 0.0
        return enriched


# ── Run simulation ──

calc   = DeltaCalculator(window=3)
engine = SAMVEDInference()

stream = [
    {'ch4_ppm': 100,  'h2s_ppm': 1.0, 'co_ppm': 8,   'o2_percent': 20.8, 'heart_rate_bpm': 73,  'spo2_percent': 98.0, 'motion_state': 1},
    {'ch4_ppm': 200,  'h2s_ppm': 2.5, 'co_ppm': 12,  'o2_percent': 20.5, 'heart_rate_bpm': 78,  'spo2_percent': 97.5, 'motion_state': 1},
    {'ch4_ppm': 600,  'h2s_ppm': 5.0, 'co_ppm': 25,  'o2_percent': 20.0, 'heart_rate_bpm': 85,  'spo2_percent': 96.8, 'motion_state': 1},
    {'ch4_ppm': 1400, 'h2s_ppm': 10,  'co_ppm': 45,  'o2_percent': 19.2, 'heart_rate_bpm': 96,  'spo2_percent': 95.0, 'motion_state': 1},
    {'ch4_ppm': 3000, 'h2s_ppm': 30,  'co_ppm': 130, 'o2_percent': 17.0, 'heart_rate_bpm': 120, 'spo2_percent': 91.0, 'motion_state': 1},
    {'ch4_ppm': 3200, 'h2s_ppm': 35,  'co_ppm': 150, 'o2_percent': 16.5, 'heart_rate_bpm': 140, 'spo2_percent': 88.0, 'motion_state': 2},
]

print("\nSAMVED – Live Stream Simulation")
print("=" * 60)
print(f"  {'t':>3s}   {'STATUS':8s}  {'RISK':>5s}  {'TREND':10s}  ACTION")
print("  " + "-" * 56)

for i, raw in enumerate(stream):
    enriched = calc.enrich(raw)
    result   = engine.predict(enriched)
    t        = i * 3
    action_short = result['action'].split('.')[0]
    flag = ''
    if result['alert_fall']:           flag = '  ⚠ FALL!'
    elif result['hard_limit_triggered']: flag = f"  ⚠ {result['override_reason']}"
    print(f"  t={t:<3d}  {result['status']:8s}  {result['risk_score']:>4d}%  {result['trend']:10s}  {action_short}{flag}")

print("\n" + "=" * 60)


SAMVED – Live Stream Simulation
    t   STATUS     RISK  TREND       ACTION
  --------------------------------------------------------
  t=0    SAFE         0%  stable      Continue work
  t=3    WARNING      0%  stable      Caution
  t=6    DANGER      51%  critical    EXIT IMMEDIATELY
  t=9    DANGER      91%  critical    EXIT IMMEDIATELY
  t=12   DANGER     100%  critical    EXIT IMMEDIATELY
  t=15   DANGER     100%  rising      EXIT IMMEDIATELY  ⚠ FALL!

